In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/pharma_sales_cleaned.csv')

df['revenue'] = df['units_sold'] * df['unit_price']

In [2]:
total_revenue = df['revenue'].sum()

total_units_sold = df['units_sold'].sum()

avg_unit_price = df['unit_price'].mean()

avg_stock_level = df['stock_level'].mean()

total_medicines = df['medicine'].nunique()

total_regions = df['region'].nunique()

In [3]:
executive_summary = pd.DataFrame({

    'Metric': [
        'Total Revenue',
        'Total Units Sold',
        'Average Unit Price',
        'Average Stock Level',
        'Unique Medicines',
        'Total Regions'
    ],

    'Value': [
        total_revenue,
        total_units_sold,
        avg_unit_price,
        avg_stock_level,
        total_medicines,
        total_regions
    ]
})

executive_summary

,Metric,Value
0,Total Revenue,4.127655e+09
1,Total Units Sold,7.853458e+07
2,Average Unit Price,5.255127e+01
3,Average Stock Level,4.553133e+03
4,Unique Medicines,1.000000e+01
5,Total Regions,8.000000e+00


In [4]:
#Region-Level KPIs
regional_dashboard = (
    df.groupby('region')
    .agg({
        'revenue': 'sum',
        'units_sold': 'sum',
        'stock_level': 'mean',
        'expiry_days_remaining': 'mean'
    })
    .reset_index()
)

regional_dashboard

,region,revenue,units_sold,stock_level,expiry_days_remaining
0,Africa,5.128403e+08,9733935,4556.151369,373.903239
1,East Asia,5.192713e+08,9837720,4551.934945,372.751825
2,Europe,5.163563e+08,9782351,4554.947308,377.691241
3,Middle East,5.156810e+08,9849229,4551.884717,372.911907
4,North America,5.151863e+08,9799919,4553.607162,374.183531
5,Oceania,5.155757e+08,9847959,4552.617290,375.930109
6,South America,5.188874e+08,9838549,4551.784945,371.372856
7,South Asia,5.138564e+08,9844922,4552.133212,374.363184


In [5]:
regional_dashboard['revenue_share_pct'] = (

    regional_dashboard['revenue']

    /

    regional_dashboard['revenue'].sum()

) * 100

In [6]:
category_dashboard = (
    df.groupby('category')
    .agg({
        'revenue': 'sum',
        'units_sold': 'sum',
        'stock_level': 'mean'
    })
    .reset_index()
)

category_dashboard

,category,revenue,units_sold,stock_level
0,Antibiotic,8.265009e+08,15679787,4554.247605
1,Antipyretic,8.295979e+08,15759748,4551.053718
2,Chronic,8.213407e+08,15687400,4553.408360
3,Cough_Cold,8.276622e+08,15679474,4554.217296
4,Vitamin,8.225529e+08,15728175,4552.736114


In [7]:
volatility = (
    df.groupby('category')['units_sold']
    .std()
    .reset_index()
)

volatility.columns = [
    'category',
    'demand_volatility'
]

In [8]:
category_dashboard = category_dashboard.merge(
    volatility,
    on='category'
)

category_dashboard

,category,revenue,units_sold,stock_level,demand_volatility
0,Antibiotic,8.265009e+08,15679787,4554.247605,329.461767
1,Antipyretic,8.295979e+08,15759748,4551.053718,344.096363
2,Chronic,8.213407e+08,15687400,4553.408360,338.283020
3,Cough_Cold,8.276622e+08,15679474,4554.217296,337.068616
4,Vitamin,8.225529e+08,15728175,4552.736114,345.302626


Time-Series Dashboard

In [9]:
#Monthly Revenue
monthly_dashboard = (
    df.groupby(['year', 'month'])['revenue']
    .sum()
    .reset_index()
)

monthly_dashboard

,year,month,revenue
0,2020,1,9.168577e+07
1,2020,2,9.558439e+07
2,2020,3,1.004198e+08
3,2020,4,1.033228e+08
4,2020,5,9.478475e+07
...,...,...,...
67,2025,8,3.310032e+07
68,2025,9,3.075167e+07
69,2025,10,3.375431e+07
70,2025,11,3.878889e+07


In [10]:
monthly_dashboard['date'] = pd.to_datetime(

    monthly_dashboard['year'].astype(str)

    +

    '-'

    +

    monthly_dashboard['month'].astype(str)

    +

    '-01'
)

In [11]:
age_dashboard = (
    df.groupby('age_group')
    .agg({
        'revenue': 'sum',
        'units_sold': 'sum'
    })
    .reset_index()
)

age_dashboard

,age_group,revenue,units_sold
0,0-12,8.219388e+08,15691865
1,13-25,8.184140e+08,15579907
2,26-45,8.207466e+08,15600149
3,46-65,8.390968e+08,15884806
4,65+,8.274585e+08,15777857


In [12]:
covid_dashboard = (
    df.groupby('covid_flag')
    .agg({
        'revenue': 'mean',
        'units_sold': 'mean'
    })
    .reset_index()
)

covid_dashboard

,covid_flag,revenue,units_sold
0,0,19553.719633,372.823075
1,1,31501.640314,597.794511


In [13]:
top_medicines = (
    df.groupby('medicine')['revenue']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

top_medicines

,medicine,revenue
0,Paracetamol,4.171531e+08
1,Azithromycin,4.167800e+08
2,Cough Syrup,4.159312e+08
3,Multivitamin,4.139161e+08
4,Metformin,4.125262e+08
5,Ibuprofen,4.124448e+08
6,Cetirizine,4.117310e+08
7,Amoxicillin,4.097209e+08
8,Amlodipine,4.088145e+08
9,Vitamin C,4.086368e+08


In [14]:
expiry_dashboard = (
    df.groupby('category')
    .agg({
        'expiry_days_remaining': 'mean',
        'stock_level': 'mean'
    })
    .reset_index()
)

expiry_dashboard

,category,expiry_days_remaining,stock_level
0,Antibiotic,374.827184,4554.247605
1,Antipyretic,373.405252,4551.053718
2,Chronic,373.774949,4553.408360
3,Cough_Cold,373.137888,4554.217296
4,Vitamin,375.547160,4552.736114


In [15]:
expiry_dashboard.to_csv(
    '../dashboard/expiry_dashboard.csv',
    index=False
)

In [14]:
top_medicines.to_csv(
    '../dashboard/top_medicines.csv',
    index=False
)

In [13]:
executive_summary.to_csv(
    '../dashboard/executive_summary.csv',
    index=False
)

In [14]:
regional_dashboard.to_csv(
    '../dashboard/regional_dashboard.csv',
    index=False
)

In [15]:
category_dashboard.to_csv(
    '../dashboard/category_dashboard.csv',
    index=False
)

In [16]:
monthly_dashboard.to_csv(
    '../dashboard/monthly_dashboard.csv',
    index=False
)

In [17]:
age_dashboard.to_csv(
    '../dashboard/age_dashboard.csv',
    index=False
)

In [18]:
covid_dashboard.to_csv(
    '../dashboard/covid_dashboard.csv',
    index=False
)